In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder,OrdinalEncoder,OneHotEncoder
from sklearn.model_selection import StratifiedKFold,RandomizedSearchCV,cross_val_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

In [2]:
df_train=pd.read_csv(r"F:\playground-series-s6e6\train.csv")
df_test=pd.read_csv(r"F:\playground-series-s6e6\test.csv")

In [3]:
df_train.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [4]:
df_test.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
1,577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
2,577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
3,577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
4,577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


In [5]:
X=df_train.drop(columns=["class"])
y=df_train["class"]
X_test=df_test.copy()

In [6]:
train_ids=X["id"]
test_ids=X_test["id"]

In [7]:
X=X.drop(columns=["id"])
X_test=X_test.drop(columns=["id"])

In [8]:
X.columns.to_list()

['alpha',
 'delta',
 'u',
 'g',
 'r',
 'i',
 'z',
 'redshift',
 'spectral_type',
 'galaxy_population']

In [9]:
X_test.columns.to_list()

['alpha',
 'delta',
 'u',
 'g',
 'r',
 'i',
 'z',
 'redshift',
 'spectral_type',
 'galaxy_population']

In [10]:
print(X.select_dtypes(include="object").columns)
print(X_test.select_dtypes(include="object").columns)

Index(['spectral_type', 'galaxy_population'], dtype='object')
Index(['spectral_type', 'galaxy_population'], dtype='object')


In [12]:
cat_cols=["spectral_type","galaxy_population"]

In [ ]:
X_ord=X.copy()
X_test_ord=X_test.copy()
ord_encoder=OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1)
X_ord[cat_cols]=ord_encoder.fit_transform(X_ord[cat_cols])
X_test_ord[cat_cols]=ord_encoder.transform(X_test_ord[cat_cols])
print(X_ord.head())

        alpha      delta          u          g          r          i  \
0  147.734256  16.959273  25.472123  21.895559  20.357926  19.257113   
1  127.988677  32.346716  20.778509  19.087062  17.587208  17.226067   
2  179.792648  35.344843  21.035203  21.079128  21.171840  20.582629   
3  225.818295  48.569421  23.305056  21.050736  19.017754  18.365658   
4  141.836135  19.342852  21.703158  19.471680  18.234449  17.899447   

           z  redshift  spectral_type  galaxy_population  
0  18.621057  0.408982            2.0                1.0  
1  16.786433  0.157976            2.0                1.0  
2  20.557366  2.823770            3.0                0.0  
3  17.914952  0.536099            2.0                1.0  
4  17.616185  0.555761            2.0                1.0  


In [ ]:
X_ohe=X.copy()
X_test_ohe=X_test.copy()

ohe=OneHotEncoder(handle_unknown="ignore",sparse_output=False)
X_cat_ohe=ohe.fit_transform(X_ohe[cat_cols])
X_test_cat_ohe=ohe.transform(X_test_ohe[cat_cols])
ohe_cols=ohe.get_feature_names_out(cat_cols)

X_cat_ohe=pd.DataFrame(X_cat_ohe,columns=ohe_cols,index=X_ohe.index)
X_test_cat_ohe=pd.DataFrame(X_test_cat_ohe,columns=ohe_cols,index=X_test_ohe.index)
X_ohe=X_ohe.drop(columns=cat_cols)
X_test_ohe=X_test_ohe.drop(columns=cat_cols)

X_ohe=pd.concat([X_ohe, X_cat_ohe], axis=1)
X_test_ohe=pd.concat([X_test_ohe, X_test_cat_ohe], axis=1)

print(X_ohe.shape)
print(X_test_ohe.shape)

X_ohe.head()

(577347, 14)
(247435, 14)


,alpha,delta,u,g,r,i,z,redshift,spectral_type_A/F,spectral_type_G/K,spectral_type_M,spectral_type_O/B,galaxy_population_Blue_Cloud,galaxy_population_Red_Sequence
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,0.0,0.0,1.0,0.0,0.0,1.0
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,0.0,0.0,1.0,0.0,0.0,1.0
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,0.0,0.0,0.0,1.0,1.0,0.0
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,0.0,0.0,1.0,0.0,0.0,1.0
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,0.0,0.0,1.0,0.0,0.0,1.0


In [15]:
print(X_ohe.columns.tolist())
print(X_ohe.shape)

['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type_A/F', 'spectral_type_G/K', 'spectral_type_M', 'spectral_type_O/B', 'galaxy_population_Blue_Cloud', 'galaxy_population_Red_Sequence']
(577347, 14)


In [18]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [19]:
target_encoder=LabelEncoder()
y_encoded=target_encoder.fit_transform(y)

In [33]:
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=len(np.unique(y_encoded)),
    eval_metric="mlogloss",
    n_estimators=700,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    colsample_bylevel=1.0,
    colsample_bynode=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    gamma=0.0,
    min_child_weight=1,
    max_delta_step=0,
    grow_policy="depthwise",
    max_leaves=0,
    max_bin=256,
    scale_pos_weight=1,
    tree_method="hist",
    device="cuda",
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

In [ ]:
scores = cross_val_score(xgb,X_ord,y_encoded,cv=cv,scoring="balanced_accuracy")
print("Mean CV:",scores.mean())
print("Std CV :",scores.std())

Mean CV: 0.9552236920790197
Std CV : 0.0005651591645743209


In [ ]:
lgbm = LGBMClassifier(
    objective="multiclass",
    num_class=len(np.unique(y_encoded)),
    boosting_type="gbdt",
    n_estimators=800,
    learning_rate=0.1,
    num_leaves=127,
    max_depth=-1,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_samples=20,
    min_child_weight=1e-3,
    max_bin=255,
    device="gpu",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

In [ ]:
lgbm_scores=cross_val_score(lgbm,X_ord,y_encoded,cv=cv,scoring="balanced_accuracy",n_jobs=-1)

Mean CV : 0.9552236920790197
Std CV  : 0.0005651591645743209


In [40]:
print("Mean CV :", lgbm_scores.mean())
print("Std CV  :", lgbm_scores.std())

Mean CV : 0.9561085898556605
Std CV  : 0.0006000838968959941


In [ ]:
cat = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="Accuracy",
    iterations=3000,
    learning_rate=0.1,
    depth=10,
    l2_leaf_reg=5,
    bootstrap_type="Bayesian",
    bagging_temperature=3,
    random_strength=2,
    grow_policy="SymmetricTree",
    task_type="GPU",
    random_seed=42,
    verbose=0
)

cat_scores = cross_val_score(cat,X,y_encoded,cv=cv,scoring="balanced_accuracy",
    fit_params={"cat_features": cat_cols})

print("CatBoost FS2")
print("Mean :", cat_scores.mean())
print("Std  :", cat_scores.std())

CatBoost FS2
Mean : 0.9507787594643038
Std  : 0.0008032088430236389


In [46]:
xgb_ohe = XGBClassifier(
    objective="multi:softprob",
    num_class=len(np.unique(y_encoded)),
    eval_metric="mlogloss",
    n_estimators=700,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    colsample_bylevel=1.0,
    colsample_bynode=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    gamma=0.0,
    min_child_weight=1,
    max_delta_step=0,
    grow_policy="depthwise",
    max_leaves=0,
    max_bin=256,
    scale_pos_weight=1,
    tree_method="hist",
    device="cuda",
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

In [47]:
xgb_ohe_scores = cross_val_score(
    xgb_ohe,
    X_ohe,
    y_encoded,
    cv=cv,
    scoring="balanced_accuracy"
)

print("Mean CV:", xgb_ohe_scores.mean())
print("Std CV :", xgb_ohe_scores.std())

Mean CV: 0.9554836714714682
Std CV : 0.0005425998228955405


In [48]:
lgbm_ohe = LGBMClassifier(
    objective="multiclass",
    num_class=len(np.unique(y_encoded)),
    boosting_type="gbdt",
    n_estimators=800,
    learning_rate=0.1,
    num_leaves=127,
    max_depth=-1,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_samples=20,
    min_child_weight=1e-3,
    max_bin=255,
    device="gpu",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

In [49]:
lgbm_scores_ohe=cross_val_score(lgbm_ohe,X_ord,y_encoded,cv=cv,scoring="balanced_accuracy",n_jobs=-1)
print("Mean CV :", lgbm_scores_ohe.mean())
print("Std CV  :", lgbm_scores_ohe.std())

Mean CV : 0.9562239922901974
Std CV  : 0.00047033934106236006
